Connected to Python 3.14.3

In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import os
import csv
import re

CITIES_FILE = "cities.txt"
DATA_FOLDER = os.path.join("Era5", "Cities")
YEAR_RANGE = range(1940, 2024)


# -----------------------------
# LOAD CITIES
# -----------------------------
def load_cities():
    cities = []

    try:
        with open(CITIES_FILE, "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                city = row.get("cities") or row.get("city")
                if city:
                    cities.append(city.strip())
    except Exception as e:
        print(f"Error loading cities: {e}")

    return cities


# -----------------------------
# PARSE FILES
# -----------------------------
def get_existing_data():
    data = {}

    if not os.path.exists(DATA_FOLDER):
        return data

    pattern = re.compile(r"ERA_(.+)_(\d{4})\.csv")

    for file in os.listdir(DATA_FOLDER):
        match = pattern.match(file)
        if match:
            city = match.group(1)
            year = int(match.group(2))

            if city not in data:
                data[city] = set()

            data[city].add(year)

    return data


# -----------------------------
# ANALYSIS
# -----------------------------
def analyze():
    cities = load_cities()
    existing = get_existing_data()

    results = []

    for city in cities:
        years_present = existing.get(city, set())
        missing_years = [y for y in YEAR_RANGE if y not in years_present]

        results.append({
            "city": city,
            "missing_count": len(missing_years),
            "missing_years": missing_years
        })

    return results


# -----------------------------
# GUI
# -----------------------------
def open_checker_window(parent):
    win = tk.Toplevel(parent)
    win.title("ERA5 Data Checker")
    win.geometry("750x450")

    frame = ttk.Frame(win)
    frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)

    columns = ("city", "missing_count")

    tree = ttk.Treeview(frame, columns=columns, show="headings")

    tree.heading("city", text="City")
    tree.heading("missing_count", text="Missing Years")

    tree.column("city", width=250)
    tree.column("missing_count", width=150)

    # --- Scrollbar ---
    scrollbar = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
    tree.configure(yscrollcommand=scrollbar.set)

    tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
    scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

    # --- Tag for missing data ---
    tree.tag_configure("missing", background="#ffcccc")  # light red

    results = analyze()

    # Store full data for click access
    city_data = {}

    for r in results:
        city = r["city"]
        missing_count = r["missing_count"]

        city_data[city] = r["missing_years"]

        tag = "missing" if missing_count > 0 else ""

        tree.insert("", tk.END, values=(city, missing_count), tags=(tag,))

    # -----------------------------
    # CLICK EVENT
    # -----------------------------
    def on_select(event):
        selected = tree.selection()
        if not selected:
            return

        item = tree.item(selected[0])
        city = item["values"][0]

        missing_years = city_data.get(city, [])

        if not missing_years:
            messagebox.showinfo("Complete", f"{city} has no missing years 🎉")
            return

        # Show full list
        years_str = ", ".join(map(str, missing_years))

        # If too long, split nicely
        if len(missing_years) > 20:
            chunks = [missing_years[i:i+10] for i in range(0, len(missing_years), 10)]
            years_str = "\n".join(", ".join(map(str, c)) for c in chunks)

        messagebox.showinfo(
            f"{city} Missing Years",
            f"Missing years ({len(missing_years)}):\n\n{years_str}"
        )

    tree.bind("<<TreeviewSelect>>", on_select)